# BTLON02 — Notebook 03: Thực nghiệm Nhóm 2 (Kiến trúc CNN nâng cao, Transfer Learning, Optimizer, Scheduler, Regularization & Fail Cases)
- **Phụ trách**: Thành viên 2 (Vai trò B) — **Nguyễn Bá Khải** (`3123411138`)
- **Môi trường**: Kaggle Notebook — bật **GPU T4**, **Internet: On** (cần để tải trọng số pretrained ImageNet), Add Input: **FER2013**.
- **Quy trình**: chạy lần lượt từng nhóm, **xem kết quả rồi mới chọn tham số cho nhóm sau** (các nhóm B → C → D phụ thuộc vào kết quả trước).
  1. **A** — Kiến trúc: `LargeCNN`, `ResNetMini`, `ResNet18/34` (pretrained), `EfficientNet-B0`, `ResNet18 frozen`
  2. **F** — Fail cases: `VGG16` 48×48 không regularization (overfit), `LargeCNN` không regularization, bản `VGG16` khắc phục
  3. **B** — Optimizer: `SGD+Momentum` vs `Adam` vs `AdamW`
  4. **C** — LR & Scheduler: `ReduceLROnPlateau` vs `CosineAnnealingLR`, quét LR
  5. **D** — Regularization: Dropout, Weight Decay, Label Smoothing
  6. **G** — Final run dài (80 epoch) → `--finalize` chọn **best model theo Val Acc** → `checkpoints/best_model.pth`
- Mỗi lần chạy **gộp** kết quả vào `logs/exp2_summary.csv`; thí nghiệm đã xong sẽ tự bỏ qua nên có thể chia nhiều phiên Kaggle.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HongHungLy/SGU26K2_HocSau_S5_DCT123C4_Nhom666HocSau.git"
BRANCH = "develop"
PROJECT_SUBDIR = "BTLON02_Emotion_Ly Hong Hung_3123411132"

if Path("/kaggle/working").exists():
    repo_dir = Path("/kaggle/working/repo")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", BRANCH], check=True)
    PROJECT_DIR = repo_dir / PROJECT_SUBDIR
else:
    cwd = Path.cwd()
    PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Working directory:", Path.cwd())

## 1. Kiểm tra GPU, dữ liệu & kiểm thử nhanh các kiến trúc

In [ ]:
import torch
from src.utils import get_device, count_parameters
from src.dataset import discover_fer2013_source
from src.models_advanced import build_model

print("PyTorch:", torch.__version__)
device = get_device()
print("FER2013 source:", discover_fer2013_source())

# Kiểm tra shape / số tham số của từng kiến trúc (không cần dữ liệu)
for arch, kw, shape in [
    ("large_cnn", dict(in_channels=1), (2, 1, 48, 48)),
    ("resnet_mini", dict(in_channels=1), (2, 1, 48, 48)),
    ("resnet18", dict(pretrained=False), (2, 3, 128, 128)),
]:
    m = build_model(arch, **kw)
    out = m(torch.randn(*shape))
    print(f"{arch:12s} out={tuple(out.shape)} params={count_parameters(m)['total']:,} last_conv={m.last_conv_layer}")

## 2. Nhóm A (Kiến trúc) + Nhóm F (Fail cases)
- `--tl_size 128` : kích thước ảnh cho Transfer Learning (128 nhanh hơn nhiều so với 224; đổi thành 224 nếu còn thời gian GPU).
- Muốn test nhanh trước: thêm `--fast_test` (2 epoch, không ghi vào bảng kết quả thật).
- Muốn bỏ thí nghiệm tùy chọn (A5 EfficientNet, F3 VGG16 scratch): thêm `--skip_optional`.

In [ ]:
!python exp2_advanced.py --group A F --tl_size 128 --num_workers 4

## 3. Xem kết quả Nhóm A + F rồi **chọn kiến trúc cho các nhóm sau**
Điền `HP_ARCH` = kiến trúc có **Val Acc cao nhất** trong Nhóm A (không chọn theo Test Acc).

In [ ]:
import pandas as pd
pd.set_option("display.width", 250)
df = pd.read_csv("logs/exp2_summary.csv")
cols = ["exp_name","total_params","best_epoch","epochs_ran","best_val_acc","test_acc","test_macro_f1","gap_at_best","final_gap","min_val_loss","train_time_sec"]
display(df[cols].sort_values("best_val_acc", ascending=False))

## 4. Nhóm B — So sánh Optimizer (SGD+Momentum vs Adam vs AdamW)

In [ ]:
HP_ARCH = "resnet_mini"     # <-- đổi theo kết quả Nhóm A: large_cnn | resnet_mini | resnet18 | ...
HP_WIDTH = 64               # <-- 64 hoặc 32 (chỉ cho resnet_mini). Nếu A2b (w32) ≈ A2 (w64) thì dùng 32 để tiết kiệm ~2-3 lần thời gian

!python exp2_advanced.py --group B --hp_arch {HP_ARCH} --hp_width {HP_WIDTH} --epochs 60

## 5. Nhóm C — Learning Rate & Scheduler (ReduceLROnPlateau vs CosineAnnealingLR)
Chọn `HP_OPT` theo optimizer tốt nhất ở Nhóm B (Val Acc).

In [ ]:
HP_OPT = "adamw"            # <-- sgd | adam | adamw
!python exp2_advanced.py --group C --hp_arch {HP_ARCH} --hp_width {HP_WIDTH} --hp_opt {HP_OPT} --epochs 60

## 6. Nhóm D — Regularization (Dropout, Weight Decay, Label Smoothing)
Chọn `HP_SCHED` và `HP_LR` (nếu khác mặc định) theo Nhóm C.

In [ ]:
HP_SCHED = "cosine"         # <-- cosine | plateau
!python exp2_advanced.py --group D --hp_arch {HP_ARCH} --hp_width {HP_WIDTH} --hp_opt {HP_OPT} --hp_sched {HP_SCHED} --epochs 60

## 7. Nhóm G — Final run (80 epoch) với tổ hợp tốt nhất rút ra từ Nhóm D
- `FINAL_DROPOUT`: none | default | strong
- `FINAL_WD`: default | high | zero | <số>
- `FINAL_LS`: 0 / 0.05 / 0.1

In [ ]:
FINAL_DROPOUT, FINAL_WD, FINAL_LS = "default", "default", 0.05   # <-- cập nhật theo kết quả Nhóm D
!python exp2_advanced.py --group G --hp_arch {HP_ARCH} --hp_width {HP_WIDTH} --hp_opt {HP_OPT} --hp_sched {HP_SCHED} \
    --final_dropout {FINAL_DROPOUT} --final_wd {FINAL_WD} --final_ls {FINAL_LS} --final_epochs 80

## 8. Chọn Best Model (theo **Val Acc**) và xuất `checkpoints/best_model.pth` cho Thành viên 3

In [ ]:
!python exp2_advanced.py --finalize

In [ ]:
from IPython.display import Image as IPyImage, display
for f in ["figures/exp2_comparison.png", "figures/exp2_fail_cases.png"]:
    if Path(f).exists():
        print(">>>", f); display(IPyImage(filename=f))
print(Path("logs/exp2_log_rows.md").read_text(encoding="utf-8"))   # dán vào Mục 2 của nhật ký thực nghiệm

## 9. Hiển thị đường cong Train/Val Loss & Accuracy của từng thực nghiệm

In [ ]:
for img_path in sorted(Path("figures").glob("Exp2_*_curves.png")):
    print(">>>", img_path.name); display(IPyImage(filename=str(img_path)))

## 10. Đóng gói kết quả để tải về
- **`tv2_advanced_outputs.zip`**: `logs/` + `figures/` (nhẹ → push lên `develop`).
- **`tv2_best_model.zip`**: chỉ `checkpoints/best_model.pth` (+ `logs/best_model_info.json`). Nếu < 95 MB: mở ngoại lệ `.gitignore` và push; nếu lớn hơn: gửi Google Drive cho TV3.
- Không push các file `.pth` khác.

In [ ]:
import zipfile
out_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

with zipfile.ZipFile(out_root / "tv2_advanced_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["figures", "logs"]:
        for f in Path(folder).rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                zf.write(f, arcname=str(f))

with zipfile.ZipFile(out_root / "tv2_best_model.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [Path("checkpoints/best_model.pth"), Path("logs/best_model_info.json")]:
        if f.exists():
            zf.write(f, arcname=str(f))
print("Đã tạo các file zip trong", out_root)